# Practice — KNN on `mpg.csv`

Same six stages as the Titanic notebook, on your own.

**Question:** given a car's specifications, was it built in the USA, Europe or Japan?

Target: `origin` &nbsp;·&nbsp; File: `mpg.csv`

In [1]:
import pandas as pd

df = pd.read_csv('mpg.csv')
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


---
## 1. Look at the data

> **Flow:** Shape, columns, what is missing.

Run `.info()` and `.shape`. Which column has missing values, and how many?

In [3]:
df.info()
print("Shape:", df.shape)
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    str    
 8   name          398 non-null    str    
dtypes: float64(4), int64(3), str(2)
memory usage: 28.1 KB
Shape: (398, 9)
mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64


How many cars from each `origin`? Use `value_counts()`.

In [4]:
print(df["origin"].value_counts())

origin
usa       249
japan      79
europe     70
Name: count, dtype: int64


---
## 2. Stage 1 — Data Cleaning

> **Flow:** Fix missing values, drop unusable columns.

Two jobs:

1. `horsepower` has blanks. Fill them with the median.
2. Drop `name`. In one line below, say why it cannot help the model.

In [5]:
# Fill missing horsepower values with the median
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

# Drop name because it is text and does not provide a useful numeric feature
df = df.drop(columns=["name"])

print(df.isnull().sum())
print(df.head())


mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model_year      0
origin          0
dtype: int64
    mpg  cylinders  displacement  horsepower  weight  acceleration  \
0  18.0          8         307.0       130.0    3504          12.0   
1  15.0          8         350.0       165.0    3693          11.5   
2  18.0          8         318.0       150.0    3436          11.0   
3  16.0          8         304.0       150.0    3433          12.0   
4  17.0          8         302.0       140.0    3449          10.5   

   model_year origin  
0          70    usa  
1          70    usa  
2          70    usa  
3          70    usa  
4          70    usa  


---
## 3. Features and Target

> **Flow:** `X` is everything the model looks at. `y` is the answer.

In [6]:
X = df.drop(columns=["origin"])
y = df["origin"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
    mpg  cylinders  displacement  horsepower  weight  acceleration  model_year
0  18.0          8         307.0       130.0    3504          12.0          70
1  15.0          8         350.0       165.0    3693          11.5          70
2  18.0          8         318.0       150.0    3436          11.0          70
3  16.0          8         304.0       150.0    3433          12.0          70
4  17.0          8         302.0       140.0    3449          10.5          70

Target:
0    usa
1    usa
2    usa
3    usa
4    usa
Name: origin, dtype: str


---
## 4. Stage 2 — Train/Test Split

> **Flow:** Hide some rows before preparing anything.

Use `test_size=0.2`, `random_state=0`, `stratify=y`.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (318, 7)
X_test: (80, 7)
y_train: (318,)
y_test: (80,)


---
## 5. Stage 3 — Feature Engineering

> **Flow:** Put every column on the same scale.

**No encoding needed here.** After dropping `name`, every feature is already a number,
so there is no text column left for `OneHotEncoder`. That happens in real projects too.

Print the min and max of each feature. Which column has the largest range?

In [8]:
print("Minimum values:")
print(X.min())

print("\nMaximum values:")
print(X.max())

print("\nRange:")
print(X.max() - X.min())


Minimum values:
mpg                9.0
cylinders          3.0
displacement      68.0
horsepower        46.0
weight          1613.0
acceleration       8.0
model_year        70.0
dtype: float64

Maximum values:
mpg               46.6
cylinders          8.0
displacement     455.0
horsepower       230.0
weight          5140.0
acceleration      24.8
model_year        82.0
dtype: float64

Range:
mpg               37.6
cylinders          5.0
displacement     387.0
horsepower       184.0
weight          3527.0
acceleration      16.8
model_year        12.0
dtype: float64


Now scale. `StandardScaler` — `fit_transform` on train, `transform` on test.

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # using only transform on testing data prevents information from the test set leaking into the model.

print("Scaled training data:")
print(X_train_scaled[:5])


Scaled training data:
[[ 1.45667302 -0.85373656 -1.15149341 -0.91814325 -1.57291589  0.89400859
  -1.37014404]
 [ 0.56019689 -0.85373656 -0.77097783 -0.41947621 -0.42282826  1.47531411
   1.59751159]
 [-1.10468733  1.49173408  1.18867744  1.2077531   0.93594462 -1.10422914
  -0.83057029]
 [ 0.56019689 -0.85373656 -0.90415828 -0.62944128 -0.93410714 -0.19593926
  -1.10035716]
 [-0.97661932  0.31899876  0.54180095 -0.1045286   0.35742272  0.89400859
  -0.83057029]]


---
## 6. Stages 4 and 5 — Train and Predict

> **Flow:** `.fit()` learns, `.predict()` answers.

**First without scaling.** Train `KNeighborsClassifier()` on the unscaled data and print the accuracy.

In [10]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_unscaled = KNeighborsClassifier()

knn_unscaled.fit(X_train, y_train)

y_pred_unscaled = knn_unscaled.predict(X_test)

accuracy_unscaled = accuracy_score(y_test, y_pred_unscaled)

print("Accuracy without scaling:", accuracy_unscaled)

Accuracy without scaling: 0.6875


**Now with scaling.** Same model, same `k`, scaled data.

In [11]:
knn_scaled = KNeighborsClassifier()

knn_scaled.fit(X_train_scaled, y_train)

y_pred_scaled = knn_scaled.predict(X_test_scaled)

accuracy_scaled = accuracy_score(y_test, y_pred_scaled)

print("Accuracy with scaling:", accuracy_scaled)

Accuracy with scaling: 0.75


---
## 7. Stage 6 — Compare

> **Flow:** Two numbers, one difference.

Print both accuracies together. Which is higher, and by how much?

In [12]:
print("Accuracy without scaling:", accuracy_unscaled)
print("Accuracy with scaling:", accuracy_scaled)

difference = accuracy_scaled - accuracy_unscaled

print("Difference:", difference)

if accuracy_scaled > accuracy_unscaled:
    print("Scaled accuracy is higher by", difference)
elif accuracy_unscaled > accuracy_scaled:
    print("Unscaled accuracy is higher by", -difference)
else:
    print("Both accuracies are the same.")

Accuracy without scaling: 0.6875
Accuracy with scaling: 0.75
Difference: 0.0625
Scaled accuracy is higher by 0.0625


*What changed between the two runs:*

The model and k stayed the same. The only change was feature scaling. 
With scaling, all features contribute more fairly to the KNN distance calculation.

---
## 8. Choosing k

> **Flow:** `k` is a dial. Try a few settings and look.

Run a loop over `k = 1, 3, 5, 7, 9, 11, 15, 21` on the **scaled** data.
Print `k` and its accuracy on each line.

In [13]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21]

results = {}

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    y_pred = knn.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    
    results[k] = accuracy
    
    print("k =", k, "Accuracy =", accuracy)

k = 1 Accuracy = 0.7375
k = 3 Accuracy = 0.75
k = 5 Accuracy = 0.75
k = 7 Accuracy = 0.75
k = 9 Accuracy = 0.75
k = 11 Accuracy = 0.725
k = 15 Accuracy = 0.7375
k = 21 Accuracy = 0.725


In [14]:
best_k = max(results, key=results.get)
best_accuracy = results[best_k]

print("Best k:", best_k)
print("Its accuracy:", best_accuracy)

Best k: 3
Its accuracy: 0.75


Does the accuracy change a lot across k, or stay roughly flat?

In [15]:
accuracies = list(results.values())

if max(accuracies) - min(accuracies) > 0.10:
    print("Accuracy changes a lot across k.")
else:
    print("Accuracy stays roughly flat across k.")

Accuracy stays roughly flat across k.
